# wandb-log-step — worked example 1: Log accuracy metric keyed by examples seen

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-log-step`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Using `examples_seen` as the x-axis for `wandb.log` (rather than batch index or epoch) makes runs with different batch sizes directly comparable on the same chart. The counter increments by `batch_size` before each log call, so the x-axis represents total training examples processed rather than iteration count.

## Worked solution

**Step 1 — initialize the counter.**
We start with `examples_seen = 0`. This counter will accumulate the total number of training examples processed across all batches.

**Step 2 — increment before logging.**
Inside the loop, we increment `examples_seen += batch_size` BEFORE calling `wandb.log`. This means the first log call has `step=batch_size`, the second has `step=2*batch_size`, etc. The logged step reflects the examples seen up to and including this batch.

**Step 3 — log the metric dict.**
We call `wandb.log({'train/accuracy': acc}, step=examples_seen)`. The metric name uses the `train/` prefix so wandb groups it in the 'train' section of the dashboard. The `step=` kwarg pins this log entry to a specific x-axis position.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def log_accuracy_loop(accuracies, batch_size):
    """
    accuracies: list of float, one per batch
    batch_size: int, examples per batch
    Returns: final examples_seen
    """
    examples_seen = 0
    for acc in accuracies:
        examples_seen += batch_size
        wandb.log({'train/accuracy': acc}, step=examples_seen)
    return examples_seen

# Exercise it
wandb.log.reset_mock()
batch_size = 32
accs = [0.42, 0.51, 0.63, 0.71]
final = log_accuracy_loop(accs, batch_size)
print('Final examples_seen:', final)  # 128
calls = wandb.log.call_args_list
for i, call in enumerate(calls):
    print(f'  call {i}: metrics={call.args[0]}, step={call.kwargs["step"]}')